# Phase 3 (Planetary Computer): Uncertainty / Confidence Map from Diffusion Sample Variance

This notebook is standalone and read-only with respect to training -- it loads an existing checkpoint and does not modify or retrain anything, so it's safe to run without affecting notebook 04's baseline or any of the Phase 1 ablation notebooks (08, 09, 10).

## What this tests

Diffusion models are probabilistic samplers, not deterministic regressors: running the sampler multiple times on the *same* input with different random noise seeds produces different outputs. That sample-to-sample variance is a standard, legitimate notion of predictive uncertainty for diffusion models -- specifically the model's **generative/output uncertainty**, not full epistemic (weight) uncertainty (which would require an ensemble of independently-trained models or MC dropout). Worth stating that distinction explicitly wherever this is written up.

This notebook:
1. Runs the sampler `N_SAMPLES` times per patch (by replicating the same conditioning across a batch, so each batch element gets an independent noise trajectory) to build an ensemble.
2. Computes the per-pixel mean (a potentially better point estimate than any single sample) and standard deviation (the confidence/uncertainty map) across that ensemble.
3. **Checks calibration**: does the predicted uncertainty (std) actually correlate with real error (|prediction - ground truth|) on held-out validation patches? This correlation, not the map's appearance alone, is the deciding evidence for whether the confidence map is a meaningful result.
4. Visualizes the uncertainty map alongside the existing GT/Pred/Error reconstruction grid for a handful of example patches.

## Which checkpoint to evaluate

Set `CHECKPOINT_NAME` below to whichever checkpoint you want to test -- the notebook-04 baseline (`s1_tuk_tessa_unet_best.pth`), or any Phase 1 ablation checkpoint (`s1_tuk_native2ch_unet_best.pth`, `s1_tuk_realattrs_unet_best.pth`, `s1_tuk_despeckled_unet_best.pth`) once trained. **Important:** the dataset class in this notebook must match whichever checkpoint you're loading -- as shipped, it matches the notebook-04 baseline (zero-filled attrs, `[VV,VH,VV,VH]` repeated channels). If you point `CHECKPOINT_NAME` at `native2ch`, `realattrs`, or `despeckled`, copy that experiment's `LidarS1Dataset` class from its own notebook into the dataset cell below first -- otherwise the data fed in won't match what the model was trained on and results will be meaningless (not just wrong, silently wrong).



**If you point `CHECKPOINT_NAME` at the `native2ch` checkpoint specifically**, the model-init cell below also needs `cond_channels_per_view=2` added to the `ConditionalUNet(...)` call (in addition to swapping the dataset class) -- otherwise the saved weights won't match the model's shape and `load_state_dict` will fail with a clear shape-mismatch error. `realattrs` and `despeckled` checkpoints don't need this, since they keep the baseline's 4-channel-per-view shape.

This notebook does not execute automatically. Run cells top to bottom on the GPU workstation.

## GPU configuration

This notebook requires a CUDA-enabled PyTorch installation.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and configuration

Must match the config used to train whichever checkpoint you're evaluating (paths, `CONTEXT_K`, `SEED`, `VAL_FRACTION`) -- this reconstructs the identical validation split. `N_SAMPLES` is the ensemble size per patch; `N_EXAMPLE_PATCHES` is how many patches get the full visual uncertainty-map figure; `N_CALIBRATION_PATCHES` is the (larger, but still much smaller than the full 251-patch validation set) sample used for the uncertainty-vs-error correlation check.

Start with `N_SAMPLES=5` for a first pass -- it's cheap and tells you whether there's any signal before committing to a larger, slower ensemble (e.g. 15-20) if the first pass looks promising.

In [ ]:

from datetime import date
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
CHECKPOINT_NAME = 's1_tuk_pcrtc_unet_best.pth'  # set to whichever of 03/04/05/06 wins -- see note above about matching the dataset class
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIDAR_SURVEY_DATE = date(2024, 4, 16)
CONTEXT_K = 3
TARGET_HW = (256, 256)
TIMESTEPS = 1000
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
N_SAMPLES = 5
N_EXAMPLE_PATCHES = 6
N_CALIBRATION_PATCHES = 40
UNCERTAINTY_TAG = CHECKPOINT_NAME.replace('s1_', '').replace('_unet_best.pth', '')


## Import Tessa's baseline implementation

The working tree intentionally does not contain Tessa's model and metrics modules, so imports use the cloned baseline path explicitly.

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddpm, p_sample_loop_ddim, p_sample_loop_plms

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

## Sentinel-1/LiDAR dataset adapter

As shipped, matches `pcrtc/03_train_baseline.ipynb`'s config (zero-filled attrs, `[VV,VH,VV,VH]` repeated channels -- note pcrtc/02's extraction already stores native 2-channel data, so 03's dataset class repeats it back to 4 to match notebook 04's contract). **Replace this cell with the matching dataset class from the winning notebook if `CHECKPOINT_NAME` above points to `pcrtc/04` (native2ch+realattrs), `pcrtc/05` (native2ch), or `pcrtc/06` (realattrs).**


In [ ]:
class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), train=False):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.train = train

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = torch.zeros(8 * self.context_k, dtype=torch.float32)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}


## Reproduce the validation split and load the checkpoint

Same pairing/shuffle logic and `SEED` as notebook 04, so this reconstructs the identical validation set the checkpoint was scored on.

In [ ]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'
random.Random(SEED).shuffle(paired_ids)
n_val = max(1, int(len(paired_ids) * VAL_FRACTION))
val_ids = paired_ids[:n_val]
print(f'Validation patches available: {len(val_ids)}')
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW, train=False)
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)  # add cond_channels_per_view=2 if CHECKPOINT_NAME is pcrtc/04 or pcrtc/05
scheduler = (LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE))
checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('Loaded checkpoint:', CHECKPOINT_NAME, '| epoch:', checkpoint.get('epoch'), '| val_loss:', checkpoint.get('val_loss'))

## Ensemble sampling function

Builds an ensemble by replicating a single patch's conditioning `N` times along the batch dimension and running the sampler once -- each batch element gets its own independent noise trajectory (`torch.randn` draws independently per batch element), so this is equivalent to N separate sampler calls but much more efficient than looping. Returns per-pixel mean (point estimate) and std (uncertainty map) in absolute elevation units (patch mean added back).

In [ ]:
def sample_ensemble(model, scheduler, sampler_fn, item, n_samples, device):
    target = item['lidar'].unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)
    condition = item['s1'].unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)
    attrs = item['attrs'].unsqueeze(0).repeat(n_samples, 1).to(device)
    with torch.no_grad():
        samples = sampler_fn(model, scheduler, target.shape, condition, attrs, device)
    mean_val = item['patch_mean'].item()
    samples_absolute = samples + mean_val  # [n_samples, 1, H, W]
    pred_mean = samples_absolute.mean(dim=0).squeeze(0)  # [H, W]
    pred_std = samples_absolute.std(dim=0).squeeze(0)    # [H, W]
    gt_absolute = item['lidar'].squeeze(0) + mean_val    # [H, W]
    return pred_mean.cpu().numpy(), pred_std.cpu().numpy(), gt_absolute.numpy(), item['mask'].numpy()

sampler = p_sample_loop_ddim

## Calibration check: does uncertainty track real error?

Runs the ensemble on `N_CALIBRATION_PATCHES` validation patches (a random subset, not the full validation set, to keep runtime reasonable), pools all valid pixels across those patches, and computes the Pearson correlation between predicted std and actual absolute error. **This correlation is the deciding evidence for whether the confidence map is a real result, not just a visualization** -- a meaningful positive correlation means high-uncertainty regions really do correspond to where the model is more often wrong.

In [ ]:
calib_ids = random.Random(SEED + 1).sample(range(len(val_dataset)), min(N_CALIBRATION_PATCHES, len(val_dataset)))

all_std = []
all_abs_error = []
calib_rows = []
for idx in calib_ids:
    item = val_dataset[idx]
    pred_mean, pred_std, gt, mask = sample_ensemble(model, scheduler, sampler, item, N_SAMPLES, DEVICE)
    mask_bool = mask.astype(bool)
    abs_error = np.abs(pred_mean - gt)
    all_std.append(pred_std[mask_bool])
    all_abs_error.append(abs_error[mask_bool])
    calib_rows.append({
        'patch_id': item['patch_id'],
        'mean_std': float(pred_std[mask_bool].mean()) if mask_bool.any() else float('nan'),
        'mean_abs_error': float(abs_error[mask_bool].mean()) if mask_bool.any() else float('nan'),
    })

all_std = np.concatenate(all_std)
all_abs_error = np.concatenate(all_abs_error)
correlation = float(np.corrcoef(all_std, all_abs_error)[0, 1])
print(f'Patches used for calibration: {len(calib_ids)} (N_SAMPLES={N_SAMPLES} each)')
print(f'Pixel-level correlation between predicted std and |error|: {correlation:.4f}')

calibration_result = {
    'checkpoint': CHECKPOINT_NAME,
    'n_samples': N_SAMPLES,
    'n_calibration_patches': len(calib_ids),
    'pixel_correlation_std_vs_abs_error': correlation,
    'per_patch': calib_rows,
}
calib_path = OUTPUT_DIR / f's1_{UNCERTAINTY_TAG}_uncertainty_calibration.json'
with calib_path.open('w') as handle:
    json.dump(calibration_result, handle, indent=2)
print('Saved:', calib_path)

## Interpreting the correlation

- **Correlation clearly positive (e.g. > 0.2-0.3) and this looks like a real, non-trivial sample size:** the confidence map is calibrated -- high predicted uncertainty really does track higher error. This is a genuine, quotable result: "the diffusion model's sample variance is a meaningful proxy for prediction reliability on this task."
- **Correlation near zero or negative:** the map isn't tracking real error at this `N_SAMPLES`. Before concluding it doesn't work, try increasing `N_SAMPLES` (e.g. to 15-20) on a smaller subset first -- a small ensemble can be too noisy to reveal a real relationship. If a larger ensemble still shows no correlation, report that honestly as a limitation rather than presenting the map as if it were validated.

Either outcome is a legitimate, reportable result for the dissertation -- the point of this check is to know which one you actually have before presenting the map.

## Qualitative uncertainty map for example patches

Extends the existing reconstruction-grid style (GT / Pred / Error / PDF) with an added Uncertainty row, for a small number of example patches. Uses a sequential colormap (`viridis`) for the std map since, unlike the signed error map, uncertainty is strictly non-negative.

In [ ]:
example_ids = list(range(min(N_EXAMPLE_PATCHES, len(val_dataset))))
examples = []
for idx in example_ids:
    item = val_dataset[idx]
    pred_mean, pred_std, gt, mask = sample_ensemble(model, scheduler, sampler, item, N_SAMPLES, DEVICE)
    examples.append({'patch_id': item['patch_id'], 'gt': gt, 'pred_mean': pred_mean, 'pred_std': pred_std, 'mask': mask})

n_cols = len(examples)
fig, axes = plt.subplots(4, n_cols, figsize=(n_cols * 4.0 + 2, 4 * 4.0), squeeze=False)
row_titles = ['GT LiDAR', 'Pred Mean', 'Error', 'Uncertainty (std)']

all_gt = np.stack([e['gt'] for e in examples])
all_pred = np.stack([e['pred_mean'] for e in examples])
max_abs_resid = np.quantile(np.abs(np.concatenate([all_gt.ravel(), all_pred.ravel()])), 0.995)
all_std_vals = np.concatenate([e['pred_std'].ravel() for e in examples])
std_vmax = np.quantile(all_std_vals, 0.99)

from matplotlib.colors import SymLogNorm
norm = SymLogNorm(linthresh=0.1, linscale=1.0, vmin=-max_abs_resid, vmax=max_abs_resid, base=10)

for col, ex in enumerate(examples):
    axes[0, col].set_title(f"Patch {ex['patch_id']}", fontsize=14, fontweight='bold')
    axes[0, col].imshow(ex['gt'], cmap='RdBu_r', norm=norm); axes[0, col].axis('off')
    axes[1, col].imshow(ex['pred_mean'], cmap='RdBu_r', norm=norm); axes[1, col].axis('off')
    err = ex['pred_mean'] - ex['gt']
    axes[2, col].imshow(err, cmap='seismic', vmin=-max_abs_resid, vmax=max_abs_resid); axes[2, col].axis('off')
    im_std = axes[3, col].imshow(ex['pred_std'], cmap='viridis', vmin=0, vmax=std_vmax); axes[3, col].axis('off')

for row in range(4):
    axes[row, 0].text(-0.25, 0.5, row_titles[row], ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')

fig.colorbar(im_std, ax=axes[3, :].tolist(), orientation='horizontal', fraction=0.05, pad=0.08, label='Std (m)')
plt.savefig(OUTPUT_DIR / f's1_{UNCERTAINTY_TAG}_uncertainty_map.png', dpi=200, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / f's1_{UNCERTAINTY_TAG}_uncertainty_map.png')
plt.show()